# ShopFloor Agent

Procedure checks for industrial assembly (MIAM / M³-HRC subset).

Flow: **order → safety → retrieve → replay → tools**.


## Setup

Helpers live in shopfloor/. Parse the FRAS subset under docs/sop_samples/.


In [ ]:
import sys
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))

from shopfloor import (
    ShopContext,
    WorkOrder,
    can_start,
    complete_step,
    is_allowed,
    parse_sop,
    retrieve,
)
from shopfloor.replay import load_session, replay
from shopfloor.tools import TOOL_SPECS, run_tool

sop = root / "docs/sop_samples/miam_assembly_subset.md"
steps = parse_sop(sop)
len(steps), steps["STEP-06"]


## Order

A step may start only if its predecessors are done.

$$\mathrm{allowed}(s) \iff \mathrm{Pred}(s) \subseteq C$$


In [ ]:
order = WorkOrder(id="WO-100", completed={"STEP-01"})
print("STEP-02", is_allowed(steps["STEP-02"], order))
print("STEP-03", is_allowed(steps["STEP-03"], order))


## Safety

High-risk steps (STEP-06 fastening) also need hands clear on the shared bench.

$$\mathrm{can\_start}(s) \iff \mathrm{allowed}(s) \land (\mathrm{risk}(s) \neq \mathrm{high} \lor H)$$


In [ ]:
order = WorkOrder(
    id="WO-310",
    completed={f"STEP-{i:02d}" for i in range(1, 6)},
)
step6 = steps["STEP-06"]

print("busy ", can_start(step6, order, ShopContext(hands_clear=False)))
print("clear", can_start(step6, order, ShopContext(hands_clear=True)))


In [ ]:
order = WorkOrder(id="WO-311")
ctx = ShopContext(hands_clear=False)

for sid in [f"STEP-{i:02d}" for i in range(1, 6)]:
    ok, why = complete_step(order, steps[sid], ctx)
    assert ok, (sid, why)

print("block", complete_step(order, step6, ctx))
ctx.hands_clear = True
print("ok   ", complete_step(order, step6, ctx))
sorted(order.completed)


## Retrieve

Lightweight SOP search before any LLM: token overlap, stopwords dropped, title boosted.


In [ ]:
for sid, score, body in retrieve(sop, "high risk fastening screwdriver", top_k=2):
    print(f"{sid}  {score:.2f}")
    print(body.splitlines()[0])
    print()

hits = retrieve(sop, "where do I put the display", top_k=2, min_score=0.2)
[(sid, round(score, 2)) for sid, score, _ in hits]


## Replay

Sample timeline in docs/sessions/sample_miam_subset.json. First STEP-06 attempt has hands busy on purpose.


In [ ]:
session = root / "docs/sessions/sample_miam_subset.json"
events = load_session(session)
findings = replay(steps, events)
[(f.t, f.step, f.reasons) for f in findings]


## Tools

Same helpers exposed for an agent. Model proposes; these decide.


In [ ]:
[t["name"] for t in TOOL_SPECS]

done = [f"STEP-{i:02d}" for i in range(1, 6)]
print(run_tool(
    "check_can_start",
    {"step_id": "STEP-06", "completed": done, "hands_clear": False},
    steps=steps,
    sop_path=sop,
))
print(run_tool(
    "retrieve_sop",
    {"query": "fastening screwdriver", "top_k": 1},
    steps=steps,
    sop_path=sop,
))


## Next

1. Swap the sample session for a real M³-HRC label export under `data/` (HF access).
2. Optional FastAPI wrapper around the same helpers.

Dataset: [M³-HRC](https://huggingface.co/datasets/ArvindSihag/M3_Multimodal_Human_Robot_Collaboration_Dataset) (CC BY-NC 4.0).
